# Chapter 14 — Shapes, Types, Devices, and Tensors

**Book alignment:** Debugging AI From First Principles, Chapter 14

**Question this notebook isolates:** `matmul` raises `(32x128 and 256x64)` on one run and a
device-mismatch `RuntimeError` on another — same training step. Does an upstream
`(shape, dtype, device)` survey at every handoff name the **first divergent boundary**, and
does one pin per run separate **H1** (wrong axis) from **H2** (wrong device)?

A "tensor" here is the triple `(shape, dtype, device)`; the ops check contracts the way a
framework would.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class T:
    shape: tuple
    dtype: str = "float32"
    device: str = "cuda:0"

def matmul(a, b):
    if a.device != b.device:
        raise RuntimeError(f"tensors on two devices, {a.device} and {b.device}")
    if a.shape[-1] != b.shape[-2]:
        raise RuntimeError(f"mat1 and mat2 shapes cannot be multiplied "
                           f"({a.shape[-2]}x{a.shape[-1]} and {b.shape[-2]}x{b.shape[-1]})")
    return T(a.shape[:-1] + (b.shape[-1],), a.dtype, a.device)

# encoder weight: expects (..., 128) @ (128, 64)

def handoff(tag, t, *, shape=None, device=None, log=None):
    if log is not None:
        log.append((tag, t.shape, t.device))
    if shape is not None and t.shape != shape:
        return (tag, f"shape {t.shape} != intended {shape}")
    if device is not None and t.device != device:
        return (tag, f"device {t.device} != intended {device}")
    return None

W_ENC = T((128, 64))                        # encoder weight

## 1. Two crashes, one training step

In [ ]:
def crash(fn):
    try:
        fn(); return "ok"
    except RuntimeError as e:
        return f"RuntimeError: {e}"

# run 1: embed output axis-swapped -> (32, 128, 256) where (32, 256, 128) was intended
embed_h1 = T((32, 128, 256))
print("run 1 (axis) :", crash(lambda: matmul(embed_h1, W_ENC)))
# run 2: axis fine, but the batch never left the CPU
embed_h2 = T((32, 256, 128), device="cpu")
print("run 2 (device):", crash(lambda: matmul(embed_h2, W_ENC)))
assert "cannot be multiplied" in crash(lambda: matmul(embed_h1, W_ENC))
assert "two devices" in crash(lambda: matmul(embed_h2, W_ENC))
print("both are RuntimeError from the same line - the text names the crash site, not the handoff")

## 2. Survey every upstream handoff — prints only, no fixes

In [ ]:
INTENDED = {"after-collate": (32, 128), "after-embed": (32, 256, 128)}

def pipeline(embed_out):
    log, first_div = [], None
    ids = T((32, 128), dtype="int64")
    for tag, t, want in [("after-collate", ids, (32, 128)), ("after-embed", embed_out, (32, 256, 128))]:
        d = handoff(tag, t, shape=want, device="cuda:0", log=log)
        if d and first_div is None:
            first_div = d
    return log, first_div

log, div = pipeline(embed_h1)
for tag, shape, device in log:
    print(f"  {tag:15} shape={shape} device={device}")
print("first divergent handoff:", div)
assert div[0] == "after-embed" and "!= intended (32, 256, 128)" in div[1]
print("the defect is a missing permute at embed->encoder, two frames above the crash")

## 3. One pin per run separates H1 from H2

In [ ]:
# H1 pin: fix the axis at the handoff (permute) -> crash clears; device pin alone would not
embed_fixed = T((32, 256, 128))
assert crash(lambda: matmul(embed_fixed, W_ENC)) == "ok"
assert crash(lambda: matmul(T((32, 128, 256), device="cpu"), W_ENC)) != "ok"   # device pin can't fix an axis bug

# H2 pin: move the operand to the device at the handoff -> that crash clears
assert crash(lambda: matmul(replace(embed_h2, device="cuda:0"), W_ENC)) == "ok"
print("axis pin clears run 1; device pin clears run 2; neither pin fixes the other's crash")
print("promote handoff('after-embed', x, shape=(32,256,128), device='cuda:0') to a permanent contract")

## What we earned

A tensor is three facts — shape, dtype, device — and every function boundary must
re-establish all three. The crashing `matmul` line was the *symptom*; the upstream survey
named `after-embed` as the first handoff whose shape diverged from intent (a missing
`permute`). One pin per run kept the experiment single-variable: the axis fix cleared the
shape crash and did nothing for the device crash, and vice versa. The catching assertion
becomes a permanent handoff contract.

**Notebook 15 / Chapter 15** takes the next silence: handoffs green, batches flowing, and
the loss curve flat as glass.